In [ ]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from tqdm import tqdm
from scipy import stats

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Configuration and paths
mac = 20
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

# ============================================================
# Variant class selection — change this to switch variant class
# ============================================================
variant_class = 'intron'
exclude_clinvar = False

# Sliding window and bootstrap parameters (for z-score evaluation)
window_size = 1_000
step_size = 10
n_boot = 1_000
max_num_variants = 100_000
target_k = np.array([1_000, 10_000], dtype=np.int64)

# Load variant class configuration
variant_class_path = "/home/dnanexus/ukbgym/config_variant_classes.yaml"
with open(variant_class_path) as f:
    variant_class_config = yaml.safe_load(f)

vc = variant_class_config[variant_class]
vc_filters = vc['variant_filtering']
selected_categories = vc['tool_categories']
x_label = f"Top N {vc['x_label']} variants"

print(f"Variant class: {variant_class}")
print(f"  Filters: {vc_filters}")
print(f"  Exclude ClinVar: {exclude_clinvar}")
print(f"  Categories: {selected_categories}")
print(f"  X-label: {x_label}")

# Load annotation configuration
config_path = "/home/dnanexus/ukbgym/config_pheno.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

# Build label/color maps for best-tissue and baseline annotations
baseline_color_map = dict(zip(anno_config_df['annotation'].to_list(), anno_config_df['color'].to_list()))
baseline_label_map = dict(zip(anno_config_df['annotation'].to_list(), anno_config_df['label'].to_list()))

best_tissue_color_map = {
    'best_tissue_corr':           '#2ca02c',   # green
    'best_tissue_zscore_all':     '#d62728',   # red
    'best_tissue_zscore_top10pct':'#9467bd',   # purple
}
best_tissue_label_map = {
    'best_tissue_corr':           'Best tissue (corr)',
    'best_tissue_zscore_all':     'Best tissue (z-all)',
    'best_tissue_zscore_top10pct':'Best tissue (z-top10%)',
}

color_map = {**best_tissue_color_map, **baseline_color_map}
label_map = {**best_tissue_label_map, **baseline_label_map}

anno_config_df

In [ ]:
# Get gene trait associations
RAP_DIR = 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'
LOCAL_DIR = '/home/dnanexus/data_dir/'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

CORR_FILE = "regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per_correlations.parquet"
!dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

loftee_corrs = (
    pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir'])
)

gene_trait_df = (
    gene_trait_df
    .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    .sort('loftee_corr_abs', descending=True)
    .unique(subset=["region"], keep="first", maintain_order=True)
)
gene_trait_df

In [ ]:
# Load AbSplice2 tissue-specific annotations
new_anno_path = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/absplice2/"
new_anno_filename = "absplice2_all_tissues_qced_maf1e-3_loftee_olink_genes_EURunrelated.parquet"
!dx download {new_anno_path}/{new_anno_filename} -o /home/dnanexus/data_dir/

new_anno_local = f'/home/dnanexus/data_dir/{new_anno_filename}'

new_anno_df = pl.scan_parquet(new_anno_local)
id_cols = ['id', 'region']
model_cols = ['pangolin_score', 'gain_score', 'loss_score', 'delta_score', 'AbSplice_DNA_max', 'AbSplice2_max']
tissue_cols = [c for c in new_anno_df.collect_schema().names() if c not in id_cols + model_cols]

new_anno_df = (
    new_anno_df
    .filter(pl.col('region').is_in(gene_trait_df['region'].unique()))
    .rename({'AbSplice2_max': 'absplice2_max'})
    .select(['id', 'region'] + tissue_cols)
    .collect()
)

new_anno_df

In [ ]:
# Load base annotations and join with AbSplice2 tissues
RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "/home/dnanexus/data_dir"
ANNO_FILE = "annotations_with_all.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")

if new_anno_local is not None:
    anno = anno.join(new_anno_df.lazy(), on=['id', 'region'], how='left')

# Build dynamic filters from variant_class.yaml
_dynamic_filters = [eval(f) for f in vc_filters]
if exclude_clinvar:
    _dynamic_filters.append(pl.col('clinical_significance').is_null())

anno = (
    anno
    .with_columns(
        encode_any_tf = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_tf', 'encode_ca-tf']),
        encode_enhancer = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_pels', 'encode_dels']),
        encode_promoter = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_pls', 'encode_ca-h3k4me3']),
    )
    .filter(
        (pl.col('region').is_in(gene_trait_df['region'].unique())),
        (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1),
        *_dynamic_filters,
    )
    .with_columns(
        promoterai_abs = pl.col('promoterai').abs(),
        promoterai_under = pl.col('promoterai'),
    )
)

baseline_annos = anno_config_df.filter(
    pl.col('category').is_in(selected_categories)
)['annotation'].to_list()

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]
selected_annos = list(set(baseline_annos).intersection(set(existing_annos)).union(set(tissue_cols)))

anno = (
    anno
    .select(set(['id', 'region']).union(set(selected_annos)))
    .collect(engine='streaming')
    .drop_nulls()
)

anno

In [ ]:
# Melt annotations into long format: (id, region, annotation, annotation_score)
melted_anno = (
    anno
    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    ).with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )
)

melted_anno

In [ ]:
# Load average phenotype per variant
RAP_APPV_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var"
LOCAL_APPV_DIR = "/home/dnanexus/data_dir"
APPV_FILE = "quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet"

!dx download {RAP_APPV_DIR}/{APPV_FILE} -o {LOCAL_APPV_DIR}/{APPV_FILE}
pheno_appv = pl.scan_parquet(f"{LOCAL_APPV_DIR}/{APPV_FILE}")

anno_keys = anno.select(pl.col('id').unique()).lazy()

pheno_appv = (
    pheno_appv
    .join(anno_keys, on='id', how='semi')
    .filter(pl.col('n_individuals') <= mac)
    .select(['id', 'phenotype', 'mean_pheno_value', 'n_individuals'])
)

---
# Train/Test Split

In [ ]:
# Per-gene 50/50 split: for each (id, region) pair independently assign to train/test
rng_split = np.random.default_rng(42)

id_region_all = anno.select(['id', 'region']).unique()
n_pairs = len(id_region_all)

train_test_split = (
    id_region_all
    .with_columns(
        is_train=pl.Series(rng_split.random(n_pairs) < 0.5)
    )
)

train_id_region_df = train_test_split.filter(pl.col('is_train')).select(['id', 'region'])
test_id_region_df = train_test_split.filter(~pl.col('is_train')).select(['id', 'region'])

print(f"Total (id, region) pairs: {n_pairs:,}")
print(f"Train pairs: {train_id_region_df.shape[0]:,}")
print(f"Test pairs: {test_id_region_df.shape[0]:,}")

---
# Best Tissue Selection — Train

## Best tissue by correlation

In [ ]:
# Filter melted_anno to train (id, region) pairs and tissue columns only
train_melted_gtex = (
    melted_anno
    .join(train_id_region_df, on=['id', 'region'], how='semi')
    .filter(pl.col('annotation').is_in(tissue_cols))
)

# Spearman correlation per (gene, tissue) on train variants
# No direction correction on mean_pheno_value — corr_beta = correlation * loftee_corr_dir handles it
train_corr_df = (
    pheno_appv
    .join(train_id_region_df.lazy(), on='id', how='inner')
    .join(gene_trait_df[["region", "phenotype"]].lazy(), on=["region", "phenotype"], how="inner")
    .join(train_melted_gtex.lazy(), on=["id", "region"], how="inner")
    .with_columns(
        pl.col(c).rank("max").over(["region", "phenotype", "annotation"]).alias(f"{c}_rank")
        for c in ['mean_pheno_value', 'annotation_score']
    )
    .group_by(["region", "phenotype", "annotation"])
    .agg(
        n_variants=pl.col("id").count(),
        correlation=pl.when(
            (pl.col("annotation_score_rank").n_unique() > 1) &
            (pl.col("mean_pheno_value_rank").n_unique() > 1)
        ).then(pl.corr("annotation_score_rank", "mean_pheno_value_rank", propagate_nans=True))
        .otherwise(None)
    )
    .join(gene_trait_df.lazy(), on=['region', 'phenotype'])
    .with_columns(corr_beta=pl.col('correlation') * pl.col('loftee_corr_dir'))
    .drop_nans().drop_nulls()
    .collect(engine='streaming')
)

# Per gene: pick tissue with highest corr_beta (min 50 train variants)
best_tissue_by_corr = (
    train_corr_df
    .filter(pl.col('n_variants') >= 50)
    .sort(['corr_beta', 'annotation'], descending=[True, False])
    .unique(subset=['region'], keep='first', maintain_order=True)
    .select(['region', pl.col('annotation').alias('best_tissue_corr'), 'corr_beta'])
)
print(f"Genes with best tissue (corr criterion): {best_tissue_by_corr.shape[0]}")
best_tissue_by_corr.head(10)

## Best tissue by avg z-score (all nonzero + top 10%)

In [ ]:
# Train variant z-scores — direction-corrected so positive = consistent with gene impairment
train_variant_zscores = (
    pheno_appv
    .join(train_id_region_df.lazy(), on='id', how='inner')
    .join(gene_trait_df[["region", "phenotype", "loftee_corr_dir"]].lazy(), on=["region", "phenotype"], how="inner")
    .with_columns(
        mean_pheno_value=pl.col('mean_pheno_value') * pl.col('loftee_corr_dir').cast(pl.Float32),
    )
    .select(['id', 'region', 'mean_pheno_value'])
    .collect(engine='streaming')
)

# Join tissue scores with z-scores, FILTER OUT zeros
train_tissue_with_z = (
    train_melted_gtex.lazy()
    .join(train_variant_zscores.lazy(), on=['id', 'region'], how='inner')
    .filter(pl.col('annotation_score') > 0)
    .with_columns(
        n_nonzero=pl.len().over(['region', 'annotation']),
        tissue_rank=pl.col('annotation_score').rank('min', descending=True).over(['region', 'annotation']),
    )
    .collect(engine='streaming')
)

# Method 1: best tissue by mean z-score of ALL nonzero-scored variants
best_tissue_by_zscore_all = (
    train_tissue_with_z.lazy()
    .group_by(['region', 'annotation'])
    .agg(mean_zscore=pl.col('mean_pheno_value').mean(), n_vars=pl.len())
    .sort(['mean_zscore', 'annotation'], descending=[True, False])
    .unique(subset=['region'], keep='first', maintain_order=True)
    .select(['region', pl.col('annotation').alias('best_tissue_zscore_all'), 'mean_zscore'])
    .collect()
)

# Method 2: best tissue by mean z-score of top 10% nonzero-scored variants
best_tissue_by_zscore_top10pct = (
    train_tissue_with_z.lazy()
    .filter(pl.col('tissue_rank') <= (pl.col('n_nonzero') * 0.1).ceil().cast(pl.UInt32).clip(lower_bound=1))
    .group_by(['region', 'annotation'])
    .agg(mean_zscore=pl.col('mean_pheno_value').mean(), n_vars=pl.len())
    .sort(['mean_zscore', 'annotation'], descending=[True, False])
    .unique(subset=['region'], keep='first', maintain_order=True)
    .select(['region', pl.col('annotation').alias('best_tissue_zscore_top10pct'), 'mean_zscore'])
    .collect()
)

print(f"Best tissue (z-all): {best_tissue_by_zscore_all.shape[0]} genes")
print(best_tissue_by_zscore_all.head(3))
print(f"\nBest tissue (z-top10%): {best_tissue_by_zscore_top10pct.shape[0]} genes")
print(best_tissue_by_zscore_top10pct.head(3))

---
# Build Test Scores

In [ ]:
# Filter melted_anno to test (id, region) pairs and tissue columns
test_melted_gtex = (
    melted_anno
    .join(test_id_region_df, on=['id', 'region'], how='semi')
    .filter(pl.col('annotation').is_in(tissue_cols))
)

# For each test variant, get its score in the gene's best tissue
best_tissue_methods = {
    'best_tissue_corr': best_tissue_by_corr,
    'best_tissue_zscore_all': best_tissue_by_zscore_all,
    'best_tissue_zscore_top10pct': best_tissue_by_zscore_top10pct,
}

test_scores = {}
for method_name, best_df in best_tissue_methods.items():
    test_scores[method_name] = (
        test_melted_gtex.lazy()
        .join(
            best_df.lazy().rename({method_name: 'annotation'}),
            on=['region', 'annotation'],
            how='inner'
        )
        .select(['id', 'region', pl.col('annotation_score').alias(method_name)])
        .collect()
    )
    print(f"{method_name}: {test_scores[method_name].shape[0]:,} test variants")

# Baseline scores from melted_anno (test pairs)
test_baseline = (
    melted_anno
    .join(test_id_region_df, on=['id', 'region'], how='semi')
    .filter(pl.col('annotation').is_in(baseline_annos))
    .pivot(index=['id', 'region'], on='annotation', values='annotation_score', aggregate_function='first')
)

# Combine all test scores into one table
test_comparison = test_scores[list(test_scores.keys())[0]].lazy()
for method_name in list(test_scores.keys())[1:]:
    test_comparison = test_comparison.join(test_scores[method_name].lazy(), on=['id', 'region'], how='full', coalesce=True)
test_comparison = test_comparison.join(test_baseline.lazy(), on=['id', 'region'], how='full', coalesce=True).collect()

print(f"\nTest comparison table: {test_comparison.shape}")
print(f"Columns: {test_comparison.columns}")
test_comparison.head()

---
# Test Evaluation — Spearman Correlation (per gene)

In [ ]:
# Annotation columns to evaluate on test
comp_cols = list(best_tissue_methods.keys()) + [c for c in baseline_annos if c in test_comparison.columns]

# Test variant phenotype values (raw — direction handled via corr_beta)
test_variant_zscores = (
    pheno_appv
    .join(test_id_region_df.lazy(), on='id', how='inner')
    .join(gene_trait_df[["region", "phenotype", "loftee_corr_dir"]].lazy(), on=["region", "phenotype"], how="inner")
    .select(['id', 'region', 'phenotype', 'mean_pheno_value', 'loftee_corr_dir'])
    .collect(engine='streaming')
)

# Spearman correlation per (gene, phenotype) for each annotation
all_test_corrs = []
for col in comp_cols:
    corr_df = (
        test_variant_zscores.lazy()
        .join(test_comparison.lazy().select(['id', 'region', col]).drop_nulls(), on=['id', 'region'], how='inner')
        .with_columns(
            anno_rank=pl.col(col).rank("average").over(["region", "phenotype"]),
            pheno_rank=pl.col('mean_pheno_value').rank("average").over(["region", "phenotype"]),
        )
        .group_by(["region", "phenotype"])
        .agg(
            n_variants=pl.col('id').count(),
            correlation=pl.corr("anno_rank", "pheno_rank", propagate_nans=True)
        )
        .join(gene_trait_df.lazy(), on=['region', 'phenotype'])
        .with_columns(
            corr_beta=pl.col('correlation') * pl.col('loftee_corr_dir'),
            annotation=pl.lit(col)
        )
        .collect()
    )
    all_test_corrs.append(corr_df)

test_corr_results = pl.concat(all_test_corrs)

# Add labels and colors for plotting
test_corr_results = test_corr_results.with_columns(
    label=pl.col('annotation').replace(label_map),
    color=pl.col('annotation').replace(color_map),
)

# Summary
test_corr_summary = (
    test_corr_results
    .drop_nans()
    .group_by('annotation')
    .agg(
        n_genes=pl.len(),
        mean_corr_beta=pl.col('corr_beta').mean(),
        median_corr_beta=pl.col('corr_beta').median(),
        se_corr_beta=pl.col('corr_beta').std() / pl.col('corr_beta').len().sqrt(),
    )
    .sort('mean_corr_beta', descending=True)
)
test_corr_summary

In [ ]:
# Correlation boxplot (pattern from pheno_correlations.ipynb)
filt_corr_df = test_corr_results.drop_nans().filter(pl.col('n_variants') > 50)

corr_pl = filt_corr_df.with_columns(
    median_corr_beta=pl.col('corr_beta').median().over("annotation")
)

ordered_labels = (
    corr_pl
    .sort("median_corr_beta", descending=False)
    .select("label")
    .unique(maintain_order=True)
    .to_series()
)
corr_pl = corr_pl.with_columns(pl.col("label").cast(pl.Enum(ordered_labels)))

color_dict = dict(corr_pl.select("annotation", "color").unique().iter_rows())

(
    ggplot(corr_pl, aes(x="label", y="corr_beta", fill="annotation"))
    + geom_hline(aes(yintercept=0), color='black', linetype='dotted')
    + geom_boxplot(alpha=0.7, outlier_shape=None)
    + theme_minimal()
    + scale_fill_manual(values=color_dict)
    + labs(
        x="",
        y="Spearman correlation",
        title=f"Best tissue evaluation — test set ({corr_pl[['region', 'phenotype']].unique().shape[0]} gene-trait pairs)"
    )
    + coord_flip()
    + theme(
        figure_size=(8, corr_pl['annotation'].n_unique() / 2 + 0.5),
        legend_position="none",
        axis_text=element_text(size=13),
        axis_title=element_text(size=13),
        plot_background=element_rect(fill="white", color="white"),
    )
)

---
# Test Evaluation — Sliding Window Z-Score (across all genes)

In [ ]:
# Build melted test scores with global ranking per annotation
# For each annotation: (id, region, annotation, annotation_score, rank_desc)

# Direction-corrected test variant z-scores
test_variant_zscores_dircor = (
    pheno_appv
    .join(test_id_region_df.lazy(), on='id', how='inner')
    .join(gene_trait_df[["region", "phenotype", "loftee_corr_dir"]].lazy(), on=["region", "phenotype"], how="inner")
    .with_columns(
        mean_pheno_value=pl.col('mean_pheno_value') * pl.col('loftee_corr_dir').cast(pl.Float32),
    )
    .select(['id', 'region', 'mean_pheno_value'])
    .collect(engine='streaming')
)

# Melt test_comparison into long format and rank globally per annotation
# Best tissue methods: AbSplice2 scores, higher = more impact (no direction correction needed)
# Baseline annotations: need direction correction via annotation_dir
melted_test_scores = []
for col in comp_cols:
    scores = test_comparison.select(['id', 'region', col]).drop_nulls().rename({col: 'annotation_score'})

    # Apply annotation direction correction for baseline annotations
    anno_dir = 1
    if col in anno_config_df['annotation'].to_list():
        anno_dir = anno_config_df.filter(pl.col('annotation') == col)['annotation_dir'][0]

    scores = scores.with_columns(
        annotation=pl.lit(col),
        annotation_score_dircor=(pl.col('annotation_score') * anno_dir).cast(pl.Float32),
    )
    melted_test_scores.append(scores)

melted_test_scores = pl.concat(melted_test_scores)

# Rank globally per annotation (descending, most damaging first)
melted_test_scores = melted_test_scores.with_columns(
    annotation_score_dircor_rank_desc=pl.col('annotation_score_dircor')
        .rank(method="max", descending=True)
        .over("annotation")
        .cast(pl.Float32)
)

# Join with z-scores, sort by rank
ranked_zscores_test = (
    test_variant_zscores_dircor.lazy()
    .join(
        melted_test_scores.lazy().select(['id', 'region', 'annotation', 'annotation_score_dircor_rank_desc']),
        on=['id', 'region'],
        how='inner'
    )
    .sort(['annotation', 'annotation_score_dircor_rank_desc'])
    .with_columns(
        row_pos=pl.col('annotation').cum_count().over('annotation'),
    )
    .pipe(lambda df: df.filter(pl.col('row_pos') <= max_num_variants) if max_num_variants is not None else df)
    .collect(engine='streaming')
)

# Map regions to integer indices for bootstrap resampling
all_regions = sorted(gene_trait_df['region'].unique().to_list())
n_regions = len(all_regions)
region_idx_map = pl.DataFrame({'region': all_regions, '_region_idx': np.arange(n_regions, dtype=np.int32)})
ranked_zscores_test = ranked_zscores_test.join(region_idx_map, on='region', how='left')

annotations_list = sorted(ranked_zscores_test['annotation'].unique().to_list())
print(f"Annotations: {annotations_list}")
print(f"Total ranked z-score rows: {ranked_zscores_test.shape[0]:,}")

In [ ]:
# --- Helper functions (from pheno_avg_zscore_gene_bootstrap.ipynb) ---

def sliding_window_means_unweighted(zscores, bin_ends, window_size):
    """Standard prefix-sum sliding window mean (for point estimates)."""
    csum = np.empty(len(zscores) + 1, dtype=np.float64)
    csum[0] = 0.0
    np.cumsum(zscores, out=csum[1:])
    return (csum[bin_ends] - csum[bin_ends - window_size]) / window_size


def expanded_csum_at(positions, cw, cwz, z, N):
    """Compute prefix sum of virtual expanded array at given positions."""
    k = np.searchsorted(cw, positions, side='left')
    k_safe = np.clip(k, 1, N)
    result = cwz[k_safe - 1] + z[k_safe - 1] * (positions - cw[k_safe - 1])
    result = np.where(positions <= 0, 0.0, result)
    return result


def sliding_window_reranked(z, cw, cwz, N, total_expanded, bin_ends, window_size):
    """Sliding window mean on the virtual expanded (re-ranked) array."""
    valid_mask = bin_ends <= total_expanded
    valid_bins = bin_ends[valid_mask]
    means = np.full(len(bin_ends), np.nan, dtype=np.float64)
    if len(valid_bins) == 0:
        return means
    right = expanded_csum_at(valid_bins, cw, cwz, z, N)
    left = expanded_csum_at(valid_bins - window_size, cw, cwz, z, N)
    means[valid_mask] = (right - left) / window_size
    return means


# --- Prepare per-annotation sorted arrays ---
anno_arrays = {}
for annotation in tqdm(annotations_list, desc='Preparing arrays'):
    adf = ranked_zscores_test.filter(pl.col('annotation') == annotation)
    z = adf['mean_pheno_value'].to_numpy().astype(np.float64)
    r_idx = adf['_region_idx'].to_numpy()
    N = len(z)
    bin_ends = np.arange(window_size, N + 1, step_size, dtype=np.int64)
    anno_arrays[annotation] = {
        'zscores': z,
        'region_idx': r_idx,
        'bin_ends': bin_ends,
        'N': N,
    }

min_variants = min(d['N'] for d in anno_arrays.values())
target_k_filt = target_k[target_k <= min_variants]
print(f"Heatmap thresholds after filtering: {target_k_filt} (min variants: {min_variants})")

# --- Point estimates ---
point_means = {}
for annotation in annotations_list:
    d = anno_arrays[annotation]
    point_means[annotation] = sliding_window_means_unweighted(d['zscores'], d['bin_ends'], window_size)

# --- Bootstrap CIs ---
rng = np.random.default_rng(42)
ci_factor = 1

boot_means = {ann: np.full((n_boot, len(anno_arrays[ann]['bin_ends'])), np.nan) for ann in annotations_list}
boot_discrete = {ann: np.full((n_boot, len(target_k_filt)), np.nan) for ann in annotations_list}

max_N = max(d['N'] for d in anno_arrays.values())
cw_buf = np.empty(max_N + 1, dtype=np.float64)
cwz_buf = np.empty(max_N + 1, dtype=np.float64)

for b in tqdm(range(n_boot), desc='Re-ranking bootstrap'):
    idx = rng.choice(n_regions, size=n_regions, replace=True)
    region_weights = np.bincount(idx, minlength=n_regions).astype(np.float64)

    for annotation in annotations_list:
        d = anno_arrays[annotation]
        N = d['N']
        z = d['zscores']
        vw = region_weights[d['region_idx']]

        cw = cw_buf[:N + 1]
        cw[0] = 0.0
        np.cumsum(vw, out=cw[1:])

        cwz = cwz_buf[:N + 1]
        cwz[0] = 0.0
        np.cumsum(z * vw, out=cwz[1:])

        total_expanded = int(cw[N])

        boot_means[annotation][b] = sliding_window_reranked(
            z, cw, cwz, N, total_expanded, d['bin_ends'], window_size
        )

        if len(target_k_filt) > 0:
            csums = expanded_csum_at(target_k_filt, cw, cwz, z, N)
            boot_discrete[annotation][b] = csums / target_k_filt

# --- Build output DataFrame ---
all_results = []
for annotation in annotations_list:
    d = anno_arrays[annotation]
    bm = boot_means[annotation]
    boot_mean = np.nanmean(bm, axis=0)
    boot_se = np.nanstd(bm, axis=0).astype(np.float32)
    all_results.append(pl.DataFrame({
        'annotation': annotation,
        'bin_end': d['bin_ends'].astype(np.float64),
        'mean_zscore': boot_mean,
        'se_zscore': boot_se,
        'ci_lower': (boot_mean - ci_factor * boot_se).astype(np.float32),
        'ci_upper': (boot_mean + ci_factor * boot_se).astype(np.float32),
    }))

zscore_binned = pl.concat(all_results).sort(['annotation', 'bin_end'])
print(f"Sliding window results: {zscore_binned.shape[0]} points across {zscore_binned['annotation'].n_unique()} annotations")

In [ ]:
# Sliding window z-score line plot (pattern from pheno_avg_zscore_gene_bootstrap.ipynb)
plt_df = (
    zscore_binned
    .drop_nans()
    .with_columns(log_bin_end=pl.col('bin_end').log10())
    .with_columns(
        label=pl.col('annotation').replace(label_map),
        color=pl.col('annotation').replace(color_map),
    )
    .sort('bin_end')
)

# Generate log-scale x-axis breaks
max_rank = zscore_binned['bin_end'].max()
min_rank = zscore_binned['bin_end'].min()
start_exp = int(np.floor(np.log10(min_rank)))
end_exp = int(np.ceil(np.log10(max_rank)))
breaks_linear = np.arange(start_exp, end_exp + 1)
labels_sci = [f"$10^{{{int(b)}}}$" for b in breaks_linear]

minor_breaks = []
for k in range(start_exp, end_exp):
    minor_breaks.extend([k + np.log10(m) for m in range(2, 10)])
minor_breaks = [b for b in minor_breaks if min_rank <= 10**b <= max_rank]

color_dict = dict(zip(plt_df['label'], plt_df['color']))

(
    ggplot(plt_df, aes(x='log_bin_end', y='mean_zscore'))
    + geom_line(aes(color='label'))
    + geom_ribbon(aes(ymin='ci_lower', ymax='ci_upper', fill='label'), alpha=0.1)
    + scale_fill_manual(values=color_dict)
    + scale_color_manual(values=color_dict)
    + labs(
        title=f"Best tissue evaluation — test set ({gene_trait_df.shape[0]} gene-trait associations)",
        subtitle=f"sliding window of {window_size} variants ({n_boot} bootstrap resamples)",
        y="Mean direction-corrected\nphenotype z-score",
        x=x_label,
        color="Annotation",
        fill="Annotation",
    )
    + scale_x_continuous(
        breaks=breaks_linear,
        labels=labels_sci,
        minor_breaks=minor_breaks,
    )
    + theme_minimal()
    + theme(
        figure_size=(7, 6),
        axis_text=element_text(size=13),
        axis_title=element_text(size=13, lineheight=1.4),
        panel_grid_minor_x=element_line(color="#e5e5e5", size=0.5),
        legend_text=element_text(size=13),
        legend_title=element_text(size=13),
        legend_position=(1, 1),
        legend_background=element_rect(fill="white", color='white', alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)

---
# Cross-Validation

In [ ]:
# CV fold assignments
n_cv_folds = 5
rng_cv = np.random.default_rng(123)

cv_fold_assignments = (
    id_region_all
    .with_columns(
        fold=pl.Series(rng_cv.integers(0, n_cv_folds, size=len(id_region_all)))
    )
)

print(f"CV folds: {n_cv_folds}")
print(cv_fold_assignments['fold'].value_counts().sort('fold'))

In [ ]:
import gc

cv_test_comparisons = []

for fold in range(n_cv_folds):
    print(f"\n--- Fold {fold+1}/{n_cv_folds} ---")
    cv_train_df = cv_fold_assignments.filter(pl.col('fold') != fold).select(['id', 'region'])
    cv_test_df  = cv_fold_assignments.filter(pl.col('fold') == fold).select(['id', 'region'])

    # ── Train: filter melted_anno to train + tissues ──
    cv_train_melted_gtex_lazy = (
        melted_anno.lazy()
        .join(cv_train_df.lazy(), on=['id', 'region'], how='semi')
        .filter(pl.col('annotation').is_in(tissue_cols))
    )

    # ── Best tissue by correlation ──
    cv_train_corr_df = (
        pheno_appv
        .join(cv_train_df.lazy(), on='id', how='inner')
        .join(gene_trait_df[["region", "phenotype"]].lazy(), on=["region", "phenotype"], how="inner")
        .join(cv_train_melted_gtex_lazy, on=["id", "region"], how="inner")
        .with_columns(
            pl.col(c).rank("max").over(["region", "phenotype", "annotation"]).alias(f"{c}_rank")
            for c in ['mean_pheno_value', 'annotation_score']
        )
        .group_by(["region", "phenotype", "annotation"])
        .agg(
            n_variants=pl.col("id").count(),
            correlation=pl.when(
                (pl.col("annotation_score_rank").n_unique() > 1) &
                (pl.col("mean_pheno_value_rank").n_unique() > 1)
            ).then(pl.corr("annotation_score_rank", "mean_pheno_value_rank", propagate_nans=True))
            .otherwise(None)
        )
        .join(gene_trait_df.lazy(), on=['region', 'phenotype'])
        .with_columns(corr_beta=pl.col('correlation') * pl.col('loftee_corr_dir'))
        .drop_nans().drop_nulls()
        .collect(engine='streaming')
    )
    cv_best_corr = (
        cv_train_corr_df
        .filter(pl.col('n_variants') >= 50)
        .sort(['corr_beta', 'annotation'], descending=[True, False])
        .unique(subset=['region'], keep='first', maintain_order=True)
        .select(['region', pl.col('annotation').alias('best_tissue_corr')])
    )
    del cv_train_corr_df

    # ── Best tissue by avg z-score ──
    cv_train_zscores = (
        pheno_appv
        .join(cv_train_df.lazy(), on='id', how='inner')
        .join(gene_trait_df[["region", "phenotype", "loftee_corr_dir"]].lazy(), on=["region", "phenotype"], how="inner")
        .with_columns(mean_pheno_value=pl.col('mean_pheno_value') * pl.col('loftee_corr_dir').cast(pl.Float32))
        .select(['id', 'region', 'mean_pheno_value'])
        .collect(engine='streaming')
    )
    cv_tissue_with_z = (
        cv_train_melted_gtex_lazy
        .join(cv_train_zscores.lazy(), on=['id', 'region'], how='inner')
        .filter(pl.col('annotation_score') > 0)
        .with_columns(
            n_nonzero=pl.len().over(['region', 'annotation']),
            tissue_rank=pl.col('annotation_score').rank('min', descending=True).over(['region', 'annotation']),
        )
        .collect(engine='streaming')
    )
    del cv_train_zscores

    cv_best_zscore_all = (
        cv_tissue_with_z.lazy()
        .group_by(['region', 'annotation'])
        .agg(mean_zscore=pl.col('mean_pheno_value').mean())
        .sort(['mean_zscore', 'annotation'], descending=[True, False])
        .unique(subset=['region'], keep='first', maintain_order=True)
        .select(['region', pl.col('annotation').alias('best_tissue_zscore_all')])
        .collect()
    )
    cv_best_zscore_top10 = (
        cv_tissue_with_z.lazy()
        .filter(pl.col('tissue_rank') <= (pl.col('n_nonzero') * 0.1).ceil().cast(pl.UInt32).clip(lower_bound=1))
        .group_by(['region', 'annotation'])
        .agg(mean_zscore=pl.col('mean_pheno_value').mean())
        .sort(['mean_zscore', 'annotation'], descending=[True, False])
        .unique(subset=['region'], keep='first', maintain_order=True)
        .select(['region', pl.col('annotation').alias('best_tissue_zscore_top10pct')])
        .collect()
    )
    del cv_tissue_with_z

    # ── Build test scores for this fold ──
    cv_test_melted_gtex_lazy = (
        melted_anno.lazy()
        .join(cv_test_df.lazy(), on=['id', 'region'], how='semi')
        .filter(pl.col('annotation').is_in(tissue_cols))
    )

    cv_best_methods = {
        'best_tissue_corr': cv_best_corr,
        'best_tissue_zscore_all': cv_best_zscore_all,
        'best_tissue_zscore_top10pct': cv_best_zscore_top10,
    }

    fold_scores = {}
    for method_name, best_df in cv_best_methods.items():
        fold_scores[method_name] = (
            cv_test_melted_gtex_lazy
            .join(best_df.lazy().rename({method_name: 'annotation'}), on=['region', 'annotation'], how='inner')
            .select(['id', 'region', pl.col('annotation_score').alias(method_name)])
            .collect()
        )
    del cv_best_corr, cv_best_zscore_all, cv_best_zscore_top10

    fold_baseline = (
        melted_anno.lazy()
        .join(cv_test_df.lazy(), on=['id', 'region'], how='semi')
        .filter(pl.col('annotation').is_in(baseline_annos))
        .collect()
        .pivot(index=['id', 'region'], on='annotation', values='annotation_score', aggregate_function='first')
    )

    fold_comparison = fold_scores[list(fold_scores.keys())[0]].lazy()
    for method_name in list(fold_scores.keys())[1:]:
        fold_comparison = fold_comparison.join(fold_scores[method_name].lazy(), on=['id', 'region'], how='full', coalesce=True)
    fold_comparison = fold_comparison.join(fold_baseline.lazy(), on=['id', 'region'], how='full', coalesce=True).collect()

    del fold_scores, fold_baseline
    cv_test_comparisons.append(fold_comparison)
    print(f"  Fold {fold+1} test rows: {fold_comparison.shape[0]:,}")
    del fold_comparison
    gc.collect()

test_comparison_cv = pl.concat(cv_test_comparisons)
del cv_test_comparisons
gc.collect()
print(f"\nCV comparison total: {test_comparison_cv.shape}")

## CV — Correlation evaluation

In [ ]:
# CV correlation evaluation — same pattern as single-split but on test_comparison_cv
comp_cols_cv = [c for c in comp_cols if c in test_comparison_cv.columns]

# All variant z-scores (entire dataset, since CV covers all folds)
id_region = anno.select(['id', 'region']).unique().lazy()
all_variant_zscores = (
    pheno_appv
    .join(id_region, on='id', how='inner')
    .join(gene_trait_df[["region", "phenotype", "loftee_corr_dir"]].lazy(), on=["region", "phenotype"], how="inner")
    .select(['id', 'region', 'phenotype', 'mean_pheno_value', 'loftee_corr_dir'])
    .collect(engine='streaming')
)

all_cv_corrs = []
for col in comp_cols_cv:
    corr_df = (
        all_variant_zscores.lazy()
        .join(test_comparison_cv.lazy().select(['id', 'region', col]).drop_nulls(), on=['id', 'region'], how='inner')
        .with_columns(
            anno_rank=pl.col(col).rank("average").over(["region", "phenotype"]),
            pheno_rank=pl.col('mean_pheno_value').rank("average").over(["region", "phenotype"]),
        )
        .group_by(["region", "phenotype"])
        .agg(
            n_variants=pl.col('id').count(),
            correlation=pl.corr("anno_rank", "pheno_rank", propagate_nans=True)
        )
        .join(gene_trait_df.lazy(), on=['region', 'phenotype'])
        .with_columns(
            corr_beta=pl.col('correlation') * pl.col('loftee_corr_dir'),
            annotation=pl.lit(col)
        )
        .collect()
    )
    all_cv_corrs.append(corr_df)

test_corr_results_cv = pl.concat(all_cv_corrs).with_columns(
    label=pl.col('annotation').replace(label_map),
    color=pl.col('annotation').replace(color_map),
)

# CV correlation boxplot
filt_cv = test_corr_results_cv.drop_nans().filter(pl.col('n_variants') > 50)
corr_cv_pl = filt_cv.with_columns(median_corr_beta=pl.col('corr_beta').median().over("annotation"))

ordered_cv_labels = (
    corr_cv_pl.sort("median_corr_beta", descending=False)
    .select("label").unique(maintain_order=True).to_series()
)
corr_cv_pl = corr_cv_pl.with_columns(pl.col("label").cast(pl.Enum(ordered_cv_labels)))
cv_color_dict = dict(corr_cv_pl.select("annotation", "color").unique().iter_rows())

(
    ggplot(corr_cv_pl, aes(x="label", y="corr_beta", fill="annotation"))
    + geom_hline(aes(yintercept=0), color='black', linetype='dotted')
    + geom_boxplot(alpha=0.7, outlier_shape=None)
    + theme_minimal()
    + scale_fill_manual(values=cv_color_dict)
    + labs(
        x="", y="Spearman correlation",
        title=f"Best tissue {n_cv_folds}-fold CV — correlation ({corr_cv_pl[['region', 'phenotype']].unique().shape[0]} gene-trait pairs)"
    )
    + coord_flip()
    + theme(
        figure_size=(8, corr_cv_pl['annotation'].n_unique() / 2 + 0.5),
        legend_position="none",
        axis_text=element_text(size=13),
        axis_title=element_text(size=13),
        plot_background=element_rect(fill="white", color="white"),
    )
)

## CV — Sliding window z-score evaluation

In [ ]:
# Direction-corrected z-scores for all variants (CV covers entire dataset)
all_variant_zscores_dircor = (
    pheno_appv
    .join(id_region, on='id', how='inner')
    .join(gene_trait_df[["region", "phenotype", "loftee_corr_dir"]].lazy(), on=["region", "phenotype"], how="inner")
    .with_columns(mean_pheno_value=pl.col('mean_pheno_value') * pl.col('loftee_corr_dir').cast(pl.Float32))
    .select(['id', 'region', 'mean_pheno_value'])
    .collect(engine='streaming')
)

# Melt CV test scores and rank globally per annotation
melted_cv_scores = []
for col in comp_cols_cv:
    scores = test_comparison_cv.select(['id', 'region', col]).drop_nulls().rename({col: 'annotation_score'})
    anno_dir = 1
    if col in anno_config_df['annotation'].to_list():
        anno_dir = anno_config_df.filter(pl.col('annotation') == col)['annotation_dir'][0]
    scores = scores.with_columns(
        annotation=pl.lit(col),
        annotation_score_dircor=(pl.col('annotation_score') * anno_dir).cast(pl.Float32),
    )
    melted_cv_scores.append(scores)

melted_cv_scores = pl.concat(melted_cv_scores).with_columns(
    annotation_score_dircor_rank_desc=pl.col('annotation_score_dircor')
        .rank(method="max", descending=True).over("annotation").cast(pl.Float32)
)

ranked_zscores_cv = (
    all_variant_zscores_dircor.lazy()
    .join(
        melted_cv_scores.lazy().select(['id', 'region', 'annotation', 'annotation_score_dircor_rank_desc']),
        on=['id', 'region'], how='inner'
    )
    .sort(['annotation', 'annotation_score_dircor_rank_desc'])
    .with_columns(row_pos=pl.col('annotation').cum_count().over('annotation'))
    .pipe(lambda df: df.filter(pl.col('row_pos') <= max_num_variants) if max_num_variants is not None else df)
    .collect(engine='streaming')
)
ranked_zscores_cv = ranked_zscores_cv.join(region_idx_map, on='region', how='left')

annotations_list_cv = sorted(ranked_zscores_cv['annotation'].unique().to_list())

# Prepare arrays and run bootstrap
anno_arrays_cv = {}
for annotation in tqdm(annotations_list_cv, desc='Preparing CV arrays'):
    adf = ranked_zscores_cv.filter(pl.col('annotation') == annotation)
    z = adf['mean_pheno_value'].to_numpy().astype(np.float64)
    r_idx = adf['_region_idx'].to_numpy()
    N = len(z)
    bin_ends = np.arange(window_size, N + 1, step_size, dtype=np.int64)
    anno_arrays_cv[annotation] = {'zscores': z, 'region_idx': r_idx, 'bin_ends': bin_ends, 'N': N}

boot_means_cv = {ann: np.full((n_boot, len(anno_arrays_cv[ann]['bin_ends'])), np.nan) for ann in annotations_list_cv}
max_N_cv = max(d['N'] for d in anno_arrays_cv.values())
cw_buf_cv = np.empty(max_N_cv + 1, dtype=np.float64)
cwz_buf_cv = np.empty(max_N_cv + 1, dtype=np.float64)

rng_cv_boot = np.random.default_rng(42)
for b in tqdm(range(n_boot), desc='CV bootstrap'):
    idx = rng_cv_boot.choice(n_regions, size=n_regions, replace=True)
    region_weights = np.bincount(idx, minlength=n_regions).astype(np.float64)
    for annotation in annotations_list_cv:
        d = anno_arrays_cv[annotation]
        N, z = d['N'], d['zscores']
        vw = region_weights[d['region_idx']]
        cw = cw_buf_cv[:N + 1]; cw[0] = 0.0; np.cumsum(vw, out=cw[1:])
        cwz = cwz_buf_cv[:N + 1]; cwz[0] = 0.0; np.cumsum(z * vw, out=cwz[1:])
        total_expanded = int(cw[N])
        boot_means_cv[annotation][b] = sliding_window_reranked(z, cw, cwz, N, total_expanded, d['bin_ends'], window_size)

all_results_cv = []
for annotation in annotations_list_cv:
    d = anno_arrays_cv[annotation]
    bm = boot_means_cv[annotation]
    boot_mean = np.nanmean(bm, axis=0)
    boot_se = np.nanstd(bm, axis=0).astype(np.float32)
    all_results_cv.append(pl.DataFrame({
        'annotation': annotation,
        'bin_end': d['bin_ends'].astype(np.float64),
        'mean_zscore': boot_mean,
        'se_zscore': boot_se,
        'ci_lower': (boot_mean - ci_factor * boot_se).astype(np.float32),
        'ci_upper': (boot_mean + ci_factor * boot_se).astype(np.float32),
    }))

zscore_binned_cv = pl.concat(all_results_cv).sort(['annotation', 'bin_end'])
print(f"CV sliding window: {zscore_binned_cv.shape[0]} points across {zscore_binned_cv['annotation'].n_unique()} annotations")

In [ ]:
# CV sliding window z-score line plot
plt_df_cv = (
    zscore_binned_cv
    .drop_nans()
    .with_columns(log_bin_end=pl.col('bin_end').log10())
    .with_columns(
        label=pl.col('annotation').replace(label_map),
        color=pl.col('annotation').replace(color_map),
    )
    .sort('bin_end')
)

max_rank_cv = zscore_binned_cv['bin_end'].max()
min_rank_cv = zscore_binned_cv['bin_end'].min()
start_exp_cv = int(np.floor(np.log10(min_rank_cv)))
end_exp_cv = int(np.ceil(np.log10(max_rank_cv)))
breaks_cv = np.arange(start_exp_cv, end_exp_cv + 1)
labels_cv = [f"$10^{{{int(b)}}}$" for b in breaks_cv]

minor_breaks_cv = []
for k in range(start_exp_cv, end_exp_cv):
    minor_breaks_cv.extend([k + np.log10(m) for m in range(2, 10)])
minor_breaks_cv = [b for b in minor_breaks_cv if min_rank_cv <= 10**b <= max_rank_cv]

color_dict_cv = dict(zip(plt_df_cv['label'], plt_df_cv['color']))

(
    ggplot(plt_df_cv, aes(x='log_bin_end', y='mean_zscore'))
    + geom_line(aes(color='label'))
    + geom_ribbon(aes(ymin='ci_lower', ymax='ci_upper', fill='label'), alpha=0.1)
    + scale_fill_manual(values=color_dict_cv)
    + scale_color_manual(values=color_dict_cv)
    + labs(
        title=f"Best tissue {n_cv_folds}-fold CV ({gene_trait_df.shape[0]} gene-trait associations)",
        subtitle=f"sliding window of {window_size} variants ({n_boot} bootstrap resamples)",
        y="Mean direction-corrected\nphenotype z-score",
        x=x_label,
        color="Annotation",
        fill="Annotation",
    )
    + scale_x_continuous(breaks=breaks_cv, labels=labels_cv, minor_breaks=minor_breaks_cv)
    + theme_minimal()
    + theme(
        figure_size=(7, 6),
        axis_text=element_text(size=13),
        axis_title=element_text(size=13, lineheight=1.4),
        panel_grid_minor_x=element_line(color="#e5e5e5", size=0.5),
        legend_text=element_text(size=13),
        legend_title=element_text(size=13),
        legend_position=(1, 1),
        legend_background=element_rect(fill="white", color='white', alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)